In [1]:
# --- 1. Import necessary libraries ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# --- 2. Load the dataset ---
# Assuming this notebook is in the 'notebooks' folder and data is in 'data' folder
df = pd.read_csv('../data/internet_service_churn.csv')

# --- 3. Data Preprocessing ---
# Drop the 'id' column as it is not a predictive feature
df_clean = df.drop(columns=['id'])

# Handle missing values based on EDA findings
# Fill missing 'reamining_contract' with 0 (assuming no contract means 0 years left)
df_clean['reamining_contract'] = df_clean['reamining_contract'].fillna(0)

# Drop rows where traffic data is missing (small percentage of data)
df_clean = df_clean.dropna(subset=['download_avg', 'upload_avg'])

# Define features (X) and the target variable (y)
X = df_clean.drop(columns=['churn'])
y = df_clean['churn']

# --- 4. Train/Test Split ---
# Split data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 5. Feature Scaling ---
# Initialize the scaler
scaler = StandardScaler()

# Define numerical columns to scale (excluding binary categorical features)
cols_to_scale = ['subscription_age', 'bill_avg', 'reamining_contract',
                 'service_failure_count', 'download_avg', 'upload_avg']

# Fit the scaler on the training data and transform both train and test data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# --- 6. Model Training ---
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# --- 7. Model Evaluation ---
# Make predictions on the test set
y_pred = rf_model.predict(X_test_scaled)

# Calculate and print evaluation metrics as required by the Technical Task
print("--- Model Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

# --- 8. Save the Model and Scaler ---
# Save the trained model and scaler to the 'src' directory for the Streamlit app
joblib.dump(rf_model, '../src/rf_model.pkl')
joblib.dump(scaler, '../src/scaler.pkl')
print("\nModel and scaler successfully saved to the '../src/' directory!")

--- Model Evaluation Metrics ---
Accuracy:  0.9423
Precision: 0.9587
Recall:    0.9368
F1 Score:  0.9477

Model and scaler successfully saved to the '../src/' directory!
